In [14]:
# ============================================================
# STEP 3 — PERCENTILE RANKS & BEHAVIOUR SCORE
# Build four percentile rank columns and a weighted behaviour
# score from the customer-level metrics.
# ============================================================

In [15]:
import pandas as pd
import numpy as np
import os

In [16]:
# Assigning path
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [17]:
# ------------------------------------------------------------
# 3A — COMPUTE PERCENTILE RANKS
# ------------------------------------------------------------
# pandas .rank(pct=True) returns each value's percentile rank
# in the population, scaled to (0, 1]. method='average' (default)
# averages tied ranks — correct for a weighted-average score.
# All four ranks are computed in ascending order: higher metric
# value → higher rank.

In [18]:
# Load the Master dataset

df_master = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)

In [19]:
# Load the Customer Aggregation dataset

df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg.pkl')
)

In [20]:
assert df_master['prices'].max() < 100, "Master pickle has corrupt prices"
print(f"Master loaded: {df_master.shape}")

# Build customer-level aggregation (overwrites whatever df_customers_agg was)
df_customers_agg = df_master.groupby('user_id').agg(
    unique_orders         = ('order_id',   'nunique'),
    unique_products       = ('product_id', 'nunique'),
    avg_product_price     = ('prices',     'mean'),
    total_reordered_items = ('reordered',  'sum'),
    total_line_items      = ('reordered',  'count')
).reset_index()

df_customers_agg['reorder_rate'] = (
    df_customers_agg['total_reordered_items'] / df_customers_agg['total_line_items']
)

# Validation
assert df_customers_agg.shape == (206209, 7), f"Wrong shape: {df_customers_agg.shape}"
assert df_customers_agg['user_id'].duplicated().sum() == 0
assert df_customers_agg.isnull().sum().sum() == 0
assert df_customers_agg['reorder_rate'].between(0, 1).all()

print(f"✅ df_customers_agg built: {df_customers_agg.shape}")
print(f"   Columns: {df_customers_agg.columns.tolist()}")

Master loaded: (32398592, 21)
✅ df_customers_agg built: (206209, 7)
   Columns: ['user_id', 'unique_orders', 'unique_products', 'avg_product_price', 'total_reordered_items', 'total_line_items', 'reorder_rate']


In [22]:
df_customers_agg['orders_pct_rank']   = df_customers_agg['unique_orders'].rank(pct=True)
df_customers_agg['reorder_pct_rank']  = df_customers_agg['reorder_rate'].rank(pct=True)
df_customers_agg['products_pct_rank'] = df_customers_agg['unique_products'].rank(pct=True)
df_customers_agg['price_pct_rank']    = df_customers_agg['avg_product_price'].rank(pct=True)

# ------------------------------------------------------------
# 3B — COMPUTE WEIGHTED BEHAVIOUR SCORE
# ------------------------------------------------------------
# Weights per the brief:
#   orders   40%  — engagement frequency (strongest signal)
#   reorder  30%  — loyalty to specific products
#   products 20%  — catalogue breadth
#   price    10%  — premium-vs-value lean
#
# DISTRIBUTION NOTE:
# Because behaviour_score is a weighted average of four percentile
# ranks, its distribution will be more centrally clustered than any
# individual rank. Most customers will fall in ~0.30–0.70. Expect
# the top tier (>= 0.80, "High Priority" in Step 4) to be a small
# minority — that's intended. The score is designed to identify
# tail behaviours meaningfully, not to spread customers uniformly.

df_customers_agg['behaviour_score'] = (
    0.40 * df_customers_agg['orders_pct_rank']
  + 0.30 * df_customers_agg['reorder_pct_rank']
  + 0.20 * df_customers_agg['products_pct_rank']
  + 0.10 * df_customers_agg['price_pct_rank']
)

# ------------------------------------------------------------
# 3C — VALIDATION
# ------------------------------------------------------------
# All ranks must be in (0, 1]
# behaviour_score must also be in (0, 1] — convex combination of ranks
# No nulls allowed in any new column

rank_cols = ['orders_pct_rank', 'reorder_pct_rank',
             'products_pct_rank', 'price_pct_rank']

for col in rank_cols:
    assert df_customers_agg[col].between(0, 1).all(), \
        f"{col} has values outside [0, 1]"
    assert df_customers_agg[col].isnull().sum() == 0, \
        f"{col} has nulls"

assert df_customers_agg['behaviour_score'].between(0, 1).all(), \
    "behaviour_score outside [0, 1] — check weights sum to 1.0"
assert df_customers_agg['behaviour_score'].isnull().sum() == 0

# Sanity: weights must sum to 1.0 (float-safe equality)
assert abs((0.40 + 0.30 + 0.20 + 0.10) - 1.0) < 1e-9

print("✅ All Step 3 assertions passed")
print()

# ------------------------------------------------------------
# 3D — DESCRIPTIVE OUTPUT
# ------------------------------------------------------------

print("Percentile rank and behaviour score distributions:")
print(df_customers_agg[rank_cols + ['behaviour_score']].describe().round(3))
print()

print("Behaviour score distribution by deciles:")
print(df_customers_agg['behaviour_score'].quantile(
    [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
).round(3))
print()

# Preview implied tier sizes for Step 4 thresholds
print("Implied tier sizes at Step 4 thresholds:")
print(f"  High Priority   (>= 0.80): {(df_customers_agg['behaviour_score'] >= 0.80).sum():>7,}  "
      f"({(df_customers_agg['behaviour_score'] >= 0.80).mean()*100:.1f}%)")
print(f"  Growth Priority (>= 0.60): {((df_customers_agg['behaviour_score'] >= 0.60) & (df_customers_agg['behaviour_score'] < 0.80)).sum():>7,}  "
      f"({((df_customers_agg['behaviour_score'] >= 0.60) & (df_customers_agg['behaviour_score'] < 0.80)).mean()*100:.1f}%)")
print(f"  Maintain        (>= 0.40): {((df_customers_agg['behaviour_score'] >= 0.40) & (df_customers_agg['behaviour_score'] < 0.60)).sum():>7,}  "
      f"({((df_customers_agg['behaviour_score'] >= 0.40) & (df_customers_agg['behaviour_score'] < 0.60)).mean()*100:.1f}%)")
print(f"  Low Priority    (<  0.40): {(df_customers_agg['behaviour_score'] < 0.40).sum():>7,}  "
      f"({(df_customers_agg['behaviour_score'] < 0.40).mean()*100:.1f}%)")
print()

print(f"✅ STEP 3 COMPLETE — df_customers_agg now has 11 columns")
print(f"   Shape: {df_customers_agg.shape}")

✅ All Step 3 assertions passed

Percentile rank and behaviour score distributions:
       orders_pct_rank  reorder_pct_rank  products_pct_rank  price_pct_rank  \
count       206209.000        206209.000         206209.000      206209.000   
mean             0.500             0.500              0.500           0.500   
std              0.288             0.289              0.289           0.289   
min              0.000             0.007              0.001           0.000   
25%              0.251             0.250              0.248           0.250   
50%              0.485             0.498              0.505           0.500   
75%              0.746             0.750              0.750           0.750   
max              0.997             1.000              1.000           1.000   

       behaviour_score  
count       206209.000  
mean             0.500  
std              0.229  
min              0.016  
25%              0.307  
50%              0.495  
75%              0.690  
max  

In [23]:
# ============================================================
# STEP 4 — PRIORITY LEVEL ASSIGNMENT
# Translate behaviour_score into a categorical priority tier.
# ============================================================

# ------------------------------------------------------------
# 4A — DEFINE TIERS WITH pd.cut
# ------------------------------------------------------------
# Bin edges: [-inf, 0.40, 0.60, 0.80, +inf]
# Using -inf and +inf guarantees that every behaviour_score lands
# in exactly one bin, regardless of floating-point edge cases.
# 
# right=False means each bin is [lower, upper), so:
#   [-inf, 0.40) → Low Priority
#   [ 0.40, 0.60) → Maintain
#   [ 0.60, 0.80) → Growth Priority
#   [ 0.80, +inf) → High Priority
#
# This matches the brief exactly:
#   "above 0.80"  = High Priority   → bin [0.80, +inf)
#   "above 0.60"  = Growth Priority → bin [0.60, 0.80)
#   "above 0.40"  = Maintain        → bin [0.40, 0.60)
#   "below 0.40"  = Low Priority    → bin [-inf, 0.40)
# Note: customers at exactly 0.40 / 0.60 / 0.80 fall into the
# upper tier (e.g. score == 0.80 → High Priority). This is the
# inclusive-lower convention agreed in the brief discussion.

import numpy as np

priority_bins   = [-np.inf, 0.40, 0.60, 0.80, np.inf]
priority_labels = ['Low Priority', 'Maintain', 'Growth Priority', 'High Priority']

df_customers_agg['priority_level'] = pd.cut(
    df_customers_agg['behaviour_score'],
    bins=priority_bins,
    labels=priority_labels,
    right=False,         # interval is [lower, upper)
    include_lowest=True  # lowest bin includes its lower edge
)

# Set categorical ordering (ascending: Low → High)
df_customers_agg['priority_level'] = df_customers_agg['priority_level'].cat.set_categories(
    priority_labels, ordered=True
)

# ------------------------------------------------------------
# 4B — VALIDATION
# ------------------------------------------------------------

# No customer left unassigned
assert df_customers_agg['priority_level'].isnull().sum() == 0, \
    "Some customers have no priority_level — bin edges are not exhaustive"

# Every category must be one of the four expected
expected = set(priority_labels)
actual   = set(df_customers_agg['priority_level'].unique())
assert actual == expected, \
    f"Unexpected categories. Expected {expected}, got {actual}"

# Cross-check: tier sizes must match the preview from Step 3
high_count   = (df_customers_agg['priority_level'] == 'High Priority').sum()
growth_count = (df_customers_agg['priority_level'] == 'Growth Priority').sum()
maintain_count = (df_customers_agg['priority_level'] == 'Maintain').sum()
low_count    = (df_customers_agg['priority_level'] == 'Low Priority').sum()

assert high_count + growth_count + maintain_count + low_count == 206209, \
    "Tier sizes don't sum to total customer count"

# Cross-check: every High Priority customer must have score >= 0.80
assert df_customers_agg.loc[df_customers_agg['priority_level'] == 'High Priority',
                            'behaviour_score'].min() >= 0.80
assert df_customers_agg.loc[df_customers_agg['priority_level'] == 'Low Priority',
                            'behaviour_score'].max() < 0.40

print("✅ All Step 4 assertions passed")
print()

# ------------------------------------------------------------
# 4C — DESCRIPTIVE OUTPUT
# ------------------------------------------------------------

print("Priority level distribution:")
print(df_customers_agg['priority_level'].value_counts().sort_index().to_string())
print()

print("Priority level distribution (%):")
print((df_customers_agg['priority_level'].value_counts(normalize=True) * 100)
      .sort_index().round(2).to_string())
print()

print("Behaviour score range per tier:")
print(df_customers_agg.groupby('priority_level', observed=True)['behaviour_score'].agg(
    ['min', 'max', 'mean', 'count']
).round(3))
print()

print(f"✅ STEP 4 COMPLETE — df_customers_agg now has 13 columns")
print(f"   Shape: {df_customers_agg.shape}")

✅ All Step 4 assertions passed

Priority level distribution:
priority_level
Low Priority       77285
Maintain           54017
Growth Priority    50055
High Priority      24852

Priority level distribution (%):
priority_level
Low Priority       37.48
Maintain           26.20
Growth Priority    24.27
High Priority      12.05

Behaviour score range per tier:
                   min    max   mean  count
priority_level                             
Low Priority     0.016  0.400  0.256  77285
Maintain         0.400  0.600  0.499  54017
Growth Priority  0.600  0.800  0.697  50055
High Priority    0.800  0.977  0.863  24852

✅ STEP 4 COMPLETE — df_customers_agg now has 13 columns
   Shape: (206209, 13)


In [24]:
# Save df_customers_agg as a checkpoint after Step 4
df_customers_agg.to_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step4.pkl')
)
print(f"✅ Checkpoint saved: customers_agg_step4.pkl")
print(f"   Shape: {df_customers_agg.shape}")
print(f"   Columns: {df_customers_agg.columns.tolist()}")

✅ Checkpoint saved: customers_agg_step4.pkl
   Shape: (206209, 13)
   Columns: ['user_id', 'unique_orders', 'unique_products', 'avg_product_price', 'total_reordered_items', 'total_line_items', 'reorder_rate', 'orders_pct_rank', 'reorder_pct_rank', 'products_pct_rank', 'price_pct_rank', 'behaviour_score', 'priority_level']
